# A guided tour of a protein

A one-minute film that tours the landmarks of a drug target, ABL kinase with the leukemia drug
imatinib (Gleevec), PDB 1IEP: the two lobes, the hinge, the gatekeeper residue, the DFG motif, the
P-loop and the drug itself. Each stop is highlighted, captioned and brought into focus.

<img src="https://raw.githubusercontent.com/denphi/speck/master/media/videos/kinase_tour.gif" width="480"/>

The film is built from a list of **stops**, so the same notebook tours your own protein: change the
PDB ID and the stops. Needs `ipyspeck>=0.8.4` and a recent Chrome, Edge, Safari (16.4+) or Firefox (130+), which make the video. Run the cells from the top, one at a time.

**Step 1: show the structure.** 1IEP holds two copies of the kinase; the tour follows chain A.

In [ ]:
from ipyspeck import Speck, shots

w = Speck.from_pdb_id("1IEP", cartoon=True)
w.apply_preset("glossy")
w

**Step 2: the stops.** One line per site: a name, the residues (a selection) and why it matters.
Residue numbers come from the structure's paper, its PDB page or UniProt; `"248-255"` is a range,
`[316, 317, 318]` a list.

In [ ]:
stops = [
    ("The hinge", {"chain": "A", "resSeq": [316, 317, 318]},
     "Glu316-Phe317-Met318 link the lobes; Met318 hydrogen-bonds the drug"),
    ("The gatekeeper, Thr315", {"chain": "A", "resSeq": 315},
     "its T315I mutation makes leukemia resistant to imatinib"),
    ("The DFG motif, flipped 'out'", {"chain": "A", "resSeq": [381, 382, 383]},
     "Asp381-Phe382-Gly383: the inactive state imatinib needs"),
    ("The P-loop", {"chain": "A", "resSeq": "248-255"},
     "the glycine-rich loop folds down over the drug"),
]

**Step 3: the film.** An overview, the two lobes, one `shots.visit` per stop, the drug, and back.
`visit` flies to a site and turns it toward you, colors it, shows its side chains, adds the caption
and pulls the focus onto it.

In [ ]:
A = {"chain": "A"}
DRUG = {"chain": "A", "resName": "STI"}      # imatinib's residue name in the PDB file
SITE = "#ff9f1c"                             # amber: the residues of each stop

film = [
    # Overview, with the title
    shots.together(shots.orbit(7, degrees=200, tilt=15),
                   shots.title("ABL kinase with imatinib",
                               subtitle="PDB 1IEP · the target of the leukemia drug Gleevec", seconds=6)),
    # Chain A; the second copy in the crystal fades
    shots.fly_to(A, 3),
    shots.crossfade(2, highlight=A, ghost=0.6, highlightScale=1.3),
    # The two lobes, colored from the N- to the C-terminus
    shots.together(shots.crossfade(1.5, cartoonColor="rainbow"), shots.rock(6, degrees=15),
                   shots.title("Two lobes", seconds=6,
                               subtitle="the N-lobe (blue) and the C-lobe (red) hold ATP, or the drug, between them")),
    shots.crossfade(1, cartoonColor="ss"),
]
# One stop per site
film += [shots.visit(site, 6, title=name, subtitle=why, color=SITE) for name, site, why in stops]
film += [
    # The drug, in its element colors
    shots.visit(DRUG, 8, title="Imatinib", color="", swing=15,
                subtitle="wedged between the lobes, touching every site on the tour"),
    # Back to the start, everything in color again
    shots.together(shots.home(5), shots.fade(5, dofStrength=0.0)),
    shots.crossfade(2, highlight={}, ghost=0.0),
    shots.hold(1),
]
print(round(shots.duration(film)), "seconds")

**Step 4: watch it, then save it.** Check a quick draft first; the final video takes a few minutes.

In [ ]:
w.preview(film)

In [ ]:
w.save_video("abl_tour_draft.mp4", film, size="720p", quality="draft")

In [ ]:
w.save_video("abl_tour.mp4", film, background={"center": "#fcfcfd", "edge": "#dde2ea"}, vignette=0.15)

### Tour your own protein

1. Change the PDB ID in step 1 (or use `Speck.from_file(...)` for your own model).
2. Replace the stops: a name, the residues, one line each. Keep captions short.
3. In step 3, change the title, the drug's residue name (`DRUG`), and drop the lobes if they do not apply.
4. Save a draft, adjust, then save the final video.

`shots.visit` options: `seconds` (5 - 8 is comfortable), `width` (Å across the picture; 30 shows a
site with its surroundings), `color` (the site's color; `""` for element colors), `focus=False`
(no depth of field) and `swing` (degrees to each side).

### If something does not work

- **Nothing happens after `save_video`**: the viewer must be on screen (scroll up to it) while the
  video is made; its progress bar shows there. The file appears when the bar finishes.
- **"cannot encode video"**: use a recent Chrome, Edge, Safari or Firefox, or save a GIF instead:
  `w.save_video("movie.gif", "spin")` (install what GIFs need with `pip install "ipyspeck[video]"`).
- **Too slow**: add `quality="draft"` for a quick test, or use `size="720p"`.
- **To stop a video being made**, press **Cancel** in the viewer.